# DETR + skip và contrastive loss · RunPod

Huấn luyện từ cùng checkpoint warm epoch 10 với notebook DETR + skip gốc.
Chạy cell cài thư viện rồi cell đăng nhập Kaggle và tải model. Checkpoint warm
được đọc từ đường dẫn KaggleHub trả về. Dataset RefCOCO và trọng số TransVG
dùng cùng luồng tải với notebook gốc.

Loss train là `8 × L1 + 2 × GIoU + 0.20 × contrastive`. Chỉnh
`CONTRASTIVE_ALPHA = 0.0` để chạy đối chứng. Mỗi run có thư mục output riêng
theo alpha. Checkpoint best được chọn bằng val Acc@0.5 toàn tập như bản gốc.

In [ ]:
%pip install -q "transformers==4.47.1" "gdown==5.2.0" kagglehub

In [ ]:
from pathlib import Path
import kagglehub

LOGIN_KAGGLE = True
WARM_HANDLE = "lamky690/epoch10-betr-baselineforfinetune/pyTorch/default"
WARM_FILENAME = None  # Điền đường dẫn tương đối nếu resource có nhiều file .pth.

if LOGIN_KAGGLE:
    kagglehub.login()

warm_dir = Path(kagglehub.model_download(WARM_HANDLE))
if WARM_FILENAME is not None:
    WARM_PATH = warm_dir / WARM_FILENAME
else:
    preferred = sorted(warm_dir.rglob("epoch_betr10.pth"))
    candidates = preferred or sorted(warm_dir.rglob("*.pth"))
    if len(candidates) != 1:
        found = "\n".join(str(p.relative_to(warm_dir)) for p in candidates)
        raise RuntimeError(f"Điền WARM_FILENAME. Tìm thấy {len(candidates)} checkpoint.\n{found}")
    WARM_PATH = candidates[0]

if not WARM_PATH.is_file():
    raise FileNotFoundError(WARM_PATH)
print("Warm checkpoint:", WARM_PATH)

In [ ]:
from pathlib import Path
import kagglehub

dataset_dir = Path(kagglehub.dataset_download("lamky690/refcoco/versions/1"))
print("Thư mục dataset:", dataset_dir)

for annotation in dataset_dir.rglob("person_samples.json"):
    print("ANNOTATION_PATH:", annotation)
    print("IMAGE_DIR:", annotation.parent / "images")

In [ ]:
from pathlib import Path
import os, json, math, time, random, hashlib, shutil, gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.ops import box_convert, generalized_box_iou_loss, roi_align
from transformers import AutoTokenizer, AutoModel, AutoConfig
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from pathlib import Path
import re



from tempfile import TemporaryDirectory

from collections import defaultdict

In [ ]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
ANNOTATION_PATH = dataset_dir / "refcoco_data" / "person_samples.json"
IMAGE_DIR = ANNOTATION_PATH.parent / "images"

# None: train mới. Resume: last_checkpoint.pth của notebook DETR này.
# Khi chuyển phiên, upload CẢ last_checkpoint.pth và best_checkpoint.pth cùng thư mục.
RESUME_PATH = None  # Nạp WARM_PATH ở epoch 10; điền last_checkpoint.pth để resume.
# Ví dụ: RESUME_PATH = Path('/workspace/checkpoints/previous-run/last_checkpoint.pth')
CONTRASTIVE_ALPHA = 0.20
TAU = 0.10
HEAD_DIM = 128
assert math.isfinite(CONTRASTIVE_ALPHA) and CONTRASTIVE_ALPHA >= 0
OUTPUT_DIR = Path(f'/workspace/rec_finetune_layer4_random_resize_skip_contrastive_a{CONTRASTIVE_ALPHA:g}')
SKIP_ALPHA = 0.10  # Fixed coefficient for this ablation. Zero runs baseline architecture.
BASELINE_BEST_PATH = None  # Optional path to baseline best_checkpoint.pth for testA.
COLOR_WORDS = ('red', 'blue', 'white', 'black', 'green', 'yellow')
assert math.isfinite(SKIP_ALPHA) and SKIP_ALPHA >= 0
assert COLOR_WORDS and all(word.strip() for word in COLOR_WORDS)
color_pattern = re.compile(
    r'\b(?:' + '|'.join(re.escape(word) for word in COLOR_WORDS) + r')\b',
    flags=re.IGNORECASE,
)
# Có thể đổi sang thư mục cache trên volume để tái sử dụng.
CACHE_ROOT = Path("/workspace/cache/rec_text")

TEXT_MODEL = 'FacebookAI/roberta-base'
DETR_MODEL = "TransVG/detr-r50-unc"
DETR_CHECKPOINT = Path('/workspace/pretrained/detr-r50-unc.pth')
DETR_DOWNLOAD_ID = '1U8Qdl-StW1yaJapnGtAEFO8Ic4X0ubtp'
AUTO_DOWNLOAD_DETR = True  # False nếu đã gắn checkpoint vào volume.
TRANSVG_VISUAL_COMMIT = 'c8624277018fc9786a39181e229f155f933b2b59' 
IMAGE_DATA_VERSION = "person_images_v1"  # Đổi nếu thay nội dung JPEG để tạo cache mới.
TEXT_REVISION = 'main'  # Lần đầu sẽ ghi commit thực tế vào cache/checkpoint.
SEED = 42
RESIZE_SCALES = (0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3)
BATCH_SIZE = 16  # Batch tổng, giữ 16 để so sánh với Kaggle
TEXT_EXTRACT_BATCH_SIZE = 128
USE_UINT8_IMAGE_CACHE = True
UINT8_CACHE_ROOT = Path('/workspace/cache/rec_uint8')
USE_CHANNELS_LAST = False
NUM_WORKERS = 10
PREFETCH_FACTOR = 2
EPOCHS = 50                      # Số epoch tối đa, kể cả khi resume.
MAX_SESSION_HOURS = None          # Tự kết thúc sau epoch để lưu output; None để tắt.
NEXT_EPOCH_RESERVE_MIN = 15.0    # Khoảng dự phòng ngoài thời gian epoch vừa đo.
L1_WEIGHT, GIOU_WEIGHT = 8.0, 2.0
OTHER_LR = 1e-4
WEIGHT_DECAY = 1e-4
LOG_EVERY = 25
RUN_TEST = False                 # Chỉ bật sau khi chốt cấu hình.
assert torch.cuda.is_available(), 'Cần GPU NVIDIA với PyTorch CUDA.'
assert ANNOTATION_PATH.is_file(), f'Sai đường dẫn dữ liệu: {ANNOTATION_PATH}'
device = torch.device('cuda:0')
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(2)
torch.backends.cudnn.benchmark = True
# benchmark có thể làm khác biệt nhỏ giữa các lượt; không bật deterministic.
print('GPU:', torch.cuda.get_device_name(device))
print('PyTorch:', str(torch.__version__))

In [ ]:
samples = json.loads(ANNOTATION_PATH.read_text(encoding='utf-8'))
train_samples = [s for s in samples if s['split'] == 'train']
val_samples = [s for s in samples if s['split'] == 'val']
test_samples = [s for s in samples if s['split'] == 'testA']
assert train_samples and val_samples and test_samples
unique_train_images = {s['image_id']: s for s in train_samples}
IMAGE_SIZE = (
    round(sum(s['height'] for s in unique_train_images.values()) / len(unique_train_images)),
    round(sum(s['width'] for s in unique_train_images.values()) / len(unique_train_images)),
)
image_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.PILToTensor(),
])
print('Train / val / testA:', len(train_samples), len(val_samples), len(test_samples))
print('Resize (H, W):', IMAGE_SIZE)

def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

annotation_sha = file_sha256(ANNOTATION_PATH)
# Kiểm tra cấu trúc dữ liệu trước khi tải model/tạo cache.
for s in samples:
    assert s['width'] > 0 and s['height'] > 0
    assert isinstance(s['text'], str) and s['text'].strip()
    assert len(s['bbox']) == 4 and s['bbox'][2] > 0 and s['bbox'][3] > 0
    assert (IMAGE_DIR / s['file_name']).is_file(), s['file_name']
print('Đã kiểm tra annotation và đường dẫn ảnh.')


## Cache RoBERTa sau 10 layer

In [ ]:
# Nếu resume, dùng lại đúng revision RoBERTa đã ghi ở lượt trước.
resume = None
if RESUME_PATH is not None:
    RESUME_PATH = Path(RESUME_PATH)
    resume = torch.load(RESUME_PATH, map_location='cpu', weights_only=True)
    cfg = resume['config']
    assert cfg['stage'] == 'rec_finetune_layer4_text2_skip_contrastive', 'Chọn checkpoint của lượt skip.'
    assert cfg['contrastive_alpha'] == CONTRASTIVE_ALPHA
    assert cfg['contrastive_tau'] == TAU and cfg['contrastive_head_dim'] == HEAD_DIM
    assert cfg['skip_alpha'] == SKIP_ALPHA, 'Giữ cùng SKIP_ALPHA khi resume.'
    assert tuple(cfg['color_words']) == COLOR_WORDS, 'Danh sách từ màu đã đổi.'
    assert cfg['annotation_sha256'] == annotation_sha, 'Dữ liệu đã thay đổi.'
    assert tuple(cfg['image_size_hw']) == IMAGE_SIZE
    if cfg['batch_size'] != BATCH_SIZE:
        print(f"Đổi batch size: {cfg['batch_size']} → {BATCH_SIZE}")
    assert cfg['loss_weights'] == [L1_WEIGHT, GIOU_WEIGHT]
    assert cfg['detr_model'] == DETR_MODEL
    assert cfg['text_model'] == TEXT_MODEL
    TEXT_REVISION = cfg['text_revision']

warm = None
if resume is None:
    try:
        warm = torch.load(WARM_PATH, map_location='cpu', weights_only=True)
    except RuntimeError as exc:
        if 'failed finding central directory' in str(exc):
            raise RuntimeError(f'Checkpoint warm-up bị lỗi archive: {WARM_PATH}') from exc
        raise
    assert warm['epoch'] == 10
    cfg = warm['config']
    assert cfg['stage'] == 'transvg_detr_r50_unc_frozen_noearlystop_v4'
    assert cfg['annotation_sha256'] == annotation_sha and tuple(cfg['image_size_hw']) == IMAGE_SIZE
    TEXT_REVISION = cfg['text_revision']

# Lấy đúng phiên bản RoBERTa từ TEXT_REVISION.
# Phần phía trên đã lấy TEXT_REVISION từ checkpoint khi resume.
config_path = hf_hub_download(
    repo_id=TEXT_MODEL,
    filename="config.json",
    revision=TEXT_REVISION,
)

# Đường dẫn cache có dạng snapshots/<commit_hash>/config.json
# Không dùng .resolve() vì file có thể là symlink sang blobs.
resolved_revision = Path(config_path).parent.name

assert re.fullmatch(r"[0-9a-fA-F]{40}", resolved_revision), (
    f"Không lấy được commit RoBERTa từ: {config_path}"
)

tokenizer = AutoTokenizer.from_pretrained(
    TEXT_MODEL,
    revision=resolved_revision,
)

print("RoBERTa revision:", resolved_revision)

all_samples = train_samples + val_samples + test_samples
unique_texts = sorted({s['text'] for s in all_samples})
text_to_index = {t:i for i,t in enumerate(unique_texts)}
text_hash = hashlib.sha256(json.dumps(unique_texts, ensure_ascii=False).encode()).hexdigest()
cache_spec = dict(text_model=TEXT_MODEL, revision=resolved_revision,
                  texts_sha256=text_hash, annotation_sha256=annotation_sha, dtype="float16", output="after_layer_10")
cache_id = hashlib.sha256(json.dumps(cache_spec, sort_keys=True).encode()).hexdigest()[:20]
CACHE_DIR = CACHE_ROOT / cache_id
manifest_path = CACHE_DIR / 'metadata.json'

if manifest_path.is_file():
    cache_meta = json.loads(manifest_path.read_text())
    assert cache_meta['spec'] == cache_spec
    assert (CACHE_DIR / 'features.npy').is_file()
    print('Tái sử dụng text cache:', CACHE_DIR)
else:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    encoded = tokenizer(unique_texts, padding=False, truncation=False)
    lengths = np.array([len(x) for x in encoded['input_ids']], dtype=np.int64)
    assert lengths.min() > 0
    assert int(lengths.max()) <= tokenizer.model_max_length
    offsets = np.concatenate([np.zeros(1, dtype=np.int64), np.cumsum(lengths)])
    required_bytes = int(offsets[-1]) * 768 * 2
    assert shutil.disk_usage(CACHE_DIR).free > required_bytes + 2 * 1024**3, 'Thiếu đĩa cho cache.'

    text_encoder = AutoModel.from_pretrained(
        TEXT_MODEL, revision=resolved_revision, add_pooling_layer=False,
    )
    text_encoder.encoder.layer = text_encoder.encoder.layer[:10]
    text_encoder.requires_grad_(False)
    text_encoder = text_encoder.to(device).eval()
    cache = np.lib.format.open_memmap(
        CACHE_DIR / 'features.npy', mode='w+', dtype=np.float16,
        shape=(int(offsets[-1]), 768),
    )
    with torch.no_grad():
        for start in tqdm(range(0, len(unique_texts), TEXT_EXTRACT_BATCH_SIZE), desc='Cache RoBERTa'):
            end = min(start + TEXT_EXTRACT_BATCH_SIZE, len(unique_texts))
            inputs = tokenizer.pad(
                {k:v[start:end] for k,v in encoded.items()},
                padding=True, return_tensors='pt',
            ).to(device)
            with torch.autocast('cuda', dtype=torch.float16):
                features = text_encoder(**inputs).last_hidden_state
            features = features.cpu().to(torch.float16).numpy()
            for j, i in enumerate(range(start, end)):
                cache[offsets[i]:offsets[i+1]] = features[j, :lengths[i]]
    cache.flush()
    np.save(CACHE_DIR / 'offsets.npy', offsets)
    cache_meta = dict(spec=cache_spec, rows=int(offsets[-1]), hidden_size=768,
                      max_length=int(lengths.max()), unique_texts=len(unique_texts))
    # Chỉ tạo manifest khi cache hoàn chỉnh.
    manifest_path.write_text(json.dumps(cache_meta, ensure_ascii=False, indent=2))
    del cache, encoded, features, inputs, text_encoder, lengths, offsets
    gc.collect()
    torch.cuda.empty_cache()
text_check = np.load(CACHE_DIR / 'features.npy', mmap_mode='r')
text_offsets = np.load(CACHE_DIR / 'offsets.npy', mmap_mode='r')
assert text_check.dtype == np.float16 and text_check.shape == (cache_meta['rows'], 768)
assert len(text_offsets) == len(unique_texts)+1 and text_offsets[-1] == len(text_check)
del text_check, text_offsets
print('Cache:', cache_meta)

In [ ]:
from pathlib import Path
import zipfile

p = Path(WARM_PATH)
print(p.resolve())
print(p.stat().st_size, "bytes")
print("ZIP đọc được:", zipfile.is_zipfile(p))

## Tải DETR và khởi tạo stem đóng băng

In [ ]:
# Copyright (c) Facebook, Inc. and its affiliates. All Rights Reserved
# Visual modules from djiajunustc/TransVG, commit:
# c8624277018fc9786a39181e229f155f933b2b59
# https://github.com/djiajunustc/TransVG/tree/main/models/visual_model
# Minimal NestedTensor utility + official backbone/PE/encoder-only execution.
import copy
import argparse
import pickle
import torchvision
from typing import Dict, List, Optional
from torch import Tensor
from torchvision.models._utils import IntermediateLayerGetter

class NestedTensor:
    def __init__(self,tensors,mask):
        self.tensors, self.mask = tensors, mask
    def decompose(self): return self.tensors,self.mask

def nested_tensor_from_tensor_list(images):
    # Tất cả ảnh trong batch cùng H,W; không padding.
    assert isinstance(images,torch.Tensor) and images.ndim == 4
    mask = torch.zeros(images.shape[0],*images.shape[-2:],device=images.device,dtype=torch.bool)
    return NestedTensor(images,mask)

class PositionEmbeddingSine(nn.Module):
    """
    This is a more standard version of the position embedding, very similar to the one
    used by the Attention is all you need paper, generalized to work on images.
    """
    def __init__(self, num_pos_feats=64, temperature=10000, normalize=False, scale=None):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, tensor_list: NestedTensor):
        x = tensor_list.tensors
        mask = tensor_list.mask
        assert mask is not None
        not_mask = ~mask
        y_embed = not_mask.cumsum(1, dtype=torch.float32)
        x_embed = not_mask.cumsum(2, dtype=torch.float32)
        if self.normalize:
            eps = 1e-6
            y_embed = y_embed / (y_embed[:, -1:, :] + eps) * self.scale
            x_embed = x_embed / (x_embed[:, :, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, :, None] / dim_t
        pos_y = y_embed[:, :, :, None] / dim_t
        pos_x = torch.stack((pos_x[:, :, :, 0::2].sin(), pos_x[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos_y = torch.stack((pos_y[:, :, :, 0::2].sin(), pos_y[:, :, :, 1::2].cos()), dim=4).flatten(3)
        pos = torch.cat((pos_y, pos_x), dim=3).permute(0, 3, 1, 2)
        return pos

class FrozenBatchNorm2d(torch.nn.Module):
    """
    BatchNorm2d where the batch statistics and the affine parameters are fixed.

    Copy-paste from torchvision.misc.ops with added eps before rqsrt,
    without which any other models than torchvision.models.resnet[18,34,50,101]
    produce nans.
    """

    def __init__(self, n):
        super(FrozenBatchNorm2d, self).__init__()
        self.register_buffer("weight", torch.ones(n))
        self.register_buffer("bias", torch.zeros(n))
        self.register_buffer("running_mean", torch.zeros(n))
        self.register_buffer("running_var", torch.ones(n))

    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict,
                              missing_keys, unexpected_keys, error_msgs):
        num_batches_tracked_key = prefix + 'num_batches_tracked'
        if num_batches_tracked_key in state_dict:
            del state_dict[num_batches_tracked_key]

        super(FrozenBatchNorm2d, self)._load_from_state_dict(
            state_dict, prefix, local_metadata, strict,
            missing_keys, unexpected_keys, error_msgs)

    def forward(self, x):
        # move reshapes to the beginning
        # to make it fuser-friendly
        w = self.weight.reshape(1, -1, 1, 1)
        b = self.bias.reshape(1, -1, 1, 1)
        rv = self.running_var.reshape(1, -1, 1, 1)
        rm = self.running_mean.reshape(1, -1, 1, 1)
        eps = 1e-5
        scale = w * (rv + eps).rsqrt()
        bias = b - rm * scale
        return x * scale + bias

class BackboneBase(nn.Module):

    def __init__(self, name:str, backbone: nn.Module, num_channels: int, return_interm_layers: bool):
        super().__init__()
        for name, parameter in backbone.named_parameters():
            if 'layer2' not in name and 'layer3' not in name and 'layer4' not in name:
                parameter.requires_grad_(False)
        if return_interm_layers:
            return_layers = {"layer1": "0", "layer2": "1", "layer3": "2", "layer4": "3"}
        else:
            return_layers = {'layer4': "0"}
        self.body = IntermediateLayerGetter(backbone, return_layers=return_layers)
        self.num_channels = num_channels

    def forward(self, tensor_list: NestedTensor):
        xs = self.body(tensor_list.tensors)
        out: Dict[str, NestedTensor] = {}
        for name, x in xs.items():
            m = tensor_list.mask
            assert m is not None
            mask = F.interpolate(m[None].float(), size=x.shape[-2:]).to(torch.bool)[0]
            out[name] = NestedTensor(x, mask)
        return out

class Backbone(BackboneBase):
    """ResNet backbone with frozen BatchNorm."""
    def __init__(self, name: str,
                 return_interm_layers: bool,
                 dilation: bool):

        backbone = getattr(torchvision.models, name)(
            replace_stride_with_dilation=[False, False, dilation],
            weights=None, norm_layer=FrozenBatchNorm2d)
            # pretrained=is_main_process(), norm_layer=FrozenBatchNorm2d)
        assert name in ('resnet50', 'resnet101')
        num_channels = 2048
        super().__init__(name, backbone, num_channels, return_interm_layers)

class Joiner(nn.Sequential):
    def __init__(self, backbone, position_embedding):
        super().__init__(backbone, position_embedding)

    def forward(self, tensor_list: NestedTensor):
        xs = self[0](tensor_list)
        out: List[NestedTensor] = []
        pos = []
        for name, x in xs.items():
            out.append(x)
            # position encoding
            pos.append(self[1](x).to(x.tensors.dtype))

        return out, pos

class Transformer(nn.Module):

    def __init__(self, d_model=512, nhead=8, num_encoder_layers=6,
                 num_decoder_layers=6, dim_feedforward=2048, dropout=0.1,
                 activation="relu", normalize_before=False,
                 return_intermediate_dec=False):
        super().__init__()

        encoder_layer = TransformerEncoderLayer(d_model, nhead, dim_feedforward,
                                                dropout, activation, normalize_before)
        encoder_norm = nn.LayerNorm(d_model) if normalize_before else None
        self.encoder = TransformerEncoder(encoder_layer, num_encoder_layers, encoder_norm)

        if num_decoder_layers > 0:
            decoder_layer = TransformerDecoderLayer(d_model, nhead, dim_feedforward,
                                                    dropout, activation, normalize_before)
            decoder_norm = nn.LayerNorm(d_model)
            self.decoder = TransformerDecoder(decoder_layer, num_decoder_layers, decoder_norm,
                                            return_intermediate=return_intermediate_dec)
        else:
            self.decoder = None

        self._reset_parameters()

        self.d_model = d_model
        self.nhead = nhead

    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, mask, pos_embed, query_embed=None):
        # flatten NxCxHxW to HWxNxC
        bs, c, h, w = src.shape
        src = src.flatten(2).permute(2, 0, 1)
        pos_embed = pos_embed.flatten(2).permute(2, 0, 1)
        mask = mask.flatten(1)
        memory = self.encoder(src, src_key_padding_mask=mask, pos=pos_embed)

        if self.decoder is not None:
            query_embed = query_embed.unsqueeze(1).repeat(1, bs, 1)
            tgt = torch.zeros_like(query_embed)
            hs = self.decoder(tgt, memory, memory_key_padding_mask=mask,
                            pos=pos_embed, query_pos=query_embed)
        
            return hs.transpose(1, 2), memory.permute(1, 2, 0).view(bs, c, h, w)
        else:
            return mask, memory

class TransformerEncoder(nn.Module):

    def __init__(self, encoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = _get_clones(encoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm

    def forward(self, src,
                mask: Optional[Tensor] = None,
                src_key_padding_mask: Optional[Tensor] = None,
                pos: Optional[Tensor] = None):
        output = src

        for layer in self.layers:
            output = layer(output, src_mask=mask,
                           src_key_padding_mask=src_key_padding_mask, pos=pos)

        if self.norm is not None:
            output = self.norm(output)

        return output

class TransformerEncoderLayer(nn.Module):

    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1,
                 activation="relu", normalize_before=False):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward_post(self,
                     src,
                     src_mask: Optional[Tensor] = None,
                     src_key_padding_mask: Optional[Tensor] = None,
                     pos: Optional[Tensor] = None):
        q = k = self.with_pos_embed(src, pos)
        src2 = self.self_attn(q, k, value=src, attn_mask=src_mask,
                              key_padding_mask=src_key_padding_mask)[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

    def forward_pre(self, src,
                    src_mask: Optional[Tensor] = None,
                    src_key_padding_mask: Optional[Tensor] = None,
                    pos: Optional[Tensor] = None):
        src2 = self.norm1(src)
        q = k = self.with_pos_embed(src2, pos)
        src2 = self.self_attn(q, k, value=src2, attn_mask=src_mask,
                              key_padding_mask=src_key_padding_mask)[0]
        src = src + self.dropout1(src2)
        src2 = self.norm2(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src2))))
        src = src + self.dropout2(src2)
        return src

    def forward(self, src,
                src_mask: Optional[Tensor] = None,
                src_key_padding_mask: Optional[Tensor] = None,
                pos: Optional[Tensor] = None):
        if self.normalize_before:
            return self.forward_pre(src, src_mask, src_key_padding_mask, pos)
        return self.forward_post(src, src_mask, src_key_padding_mask, pos)

def _get_clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])

def _get_activation_fn(activation):
    """Return an activation function given a string"""
    if activation == "relu":
        return F.relu
    if activation == "gelu":
        return F.gelu
    if activation == "glu":
        return F.glu
    raise RuntimeError(F"activation should be relu/gelu, not {activation}.")

class DETR(nn.Module):
    """ This is the DETR module that performs object detection """
    def __init__(self, backbone, transformer, num_queries, train_backbone, train_transformer, aux_loss=False):
        """ Initializes the model.
        Parameters:
            backbone: torch module of the backbone to be used. See backbone.py
            transformer: torch module of the transformer architecture. See transformer.py
            num_classes: number of object classes
            num_queries: number of object queries, ie detection slot. This is the maximal number of objects
                         DETR can detect in a single image. For COCO, we recommend 100 queries.
            aux_loss: True if auxiliary decoding losses (loss at each decoder layer) are to be used.
        """
        super().__init__()
        self.num_queries = num_queries
        self.transformer = transformer
        self.backbone = backbone

        if self.transformer is not None:
            hidden_dim = transformer.d_model
            self.input_proj = nn.Conv2d(backbone.num_channels, hidden_dim, kernel_size=1)
        else:
            hidden_dim = backbone.num_channels

        if not train_backbone:
            for p in self.backbone.parameters():
                p.requires_grad_(False)
        
        if self.transformer is not None and not train_transformer:
            for m in [self.transformer, self.input_proj]:
                for p in m.parameters():
                    p.requires_grad_(False)

        self.num_channels = hidden_dim

    def forward(self, samples: NestedTensor):
        """ The forward expects a NestedTensor, which consists of:
               - samples.tensor: batched images, of shape [batch_size x 3 x H x W]
               - samples.mask: a binary mask of shape [batch_size x H x W], containing 1 on padded pixels

            It returns a dict with the following elements:
               - "pred_logits": the classification logits (including no-object) for all queries.
                                Shape= [batch_size x num_queries x (num_classes + 1)]
               - "pred_boxes": The normalized boxes coordinates for all queries, represented as
                               (center_x, center_y, height, width). These values are normalized in [0, 1],
                               relative to the size of each individual image (disregarding possible padding).
                               See PostProcess for information on how to retrieve the unnormalized bounding box.
               - "aux_outputs": Optional, only returned when auxilary losses are activated. It is a list of
                                dictionnaries containing the two above keys for each decoder layer.
        """
        if isinstance(samples, (list, torch.Tensor)):
            samples = nested_tensor_from_tensor_list(samples)
        features, pos = self.backbone(samples)

        src, mask = features[-1].decompose()
        assert mask is not None

        if self.transformer is not None:
            out = self.transformer(self.input_proj(src), mask, pos[-1], query_embed=None)
        else:
            out = [mask.flatten(1), src.flatten(2).permute(2, 0, 1)]
             
        return out

class FrozenDETRFeatures(nn.Module):
    def __init__(self,checkpoint):
        super().__init__()
        backbone = Backbone('resnet50', return_interm_layers=False, dilation=False)
        position = PositionEmbeddingSine(128, normalize=True)
        joiner = Joiner(backbone,position)
        joiner.num_channels = backbone.num_channels
        transformer = Transformer(d_model=256,nhead=8,num_encoder_layers=6,
                                  num_decoder_layers=0,dim_feedforward=2048,
                                  dropout=0.1,activation='relu',normalize_before=False)
        self.visual = DETR(joiner,transformer,num_queries=100,
                           train_backbone=False,train_transformer=False)
        # Một số checkpoint gốc chứa argparse.Namespace; chỉ allowlist lớp đó.
        with torch.serialization.safe_globals([argparse.Namespace]):
            payload = torch.load(checkpoint,map_location='cpu',weights_only=True)
        state = payload.get('model',payload)
        assert isinstance(state,dict) and state, 'Checkpoint thiếu model state_dict.'
        state = {k.removeprefix('module.'):v for k,v in state.items()}
        required = self.visual.state_dict()
        missing = sorted(set(required)-set(state))
        assert not missing, f'Checkpoint thiếu visual weights: {missing[:20]}'
        mismatched = [k for k in required if tuple(required[k].shape)!=tuple(state[k].shape)]
        assert not mismatched, f'Backbone/cấu hình checkpoint không khớp: {mismatched[:20]}'
        extra = sorted(set(state)-set(required))
        # Chỉ bỏ detection decoder/head/query; không bỏ nhầm encoder/backbone.
        allowed = ('transformer.decoder.','class_embed.','bbox_embed.','query_embed.')
        unexpected = [k for k in extra if not k.startswith(allowed)]
        assert not unexpected, f'Checkpoint có visual keys không nhận diện: {unexpected[:20]}'
        self.visual.load_state_dict({k:state[k] for k in required},strict=True)
        print('Loaded ALL backbone + input_proj + 6-layer encoder weights;',
              len(extra),'unused detection decoder/head keys.')
        del state,payload
        self.requires_grad_(False)
        self.eval()
    def train(self,mode=True):
        super().train(False)
        return self
    @torch.no_grad()
    @torch.autocast(device_type='cuda',dtype=torch.float16)
    def forward(self,images):
        padding,memory = self.visual(nested_tensor_from_tensor_list(images))
        h,w = math.ceil(images.shape[-2]/32),math.ceil(images.shape[-1]/32)
        assert tuple(memory.shape) == (h*w,len(images),256)
        assert not padding.any(), 'Ảnh đã resize cùng size nên mask phải toàn False.'
        return memory.permute(1,2,0).reshape(len(images),256,h,w).contiguous()


In [ ]:
assert DETR_CHECKPOINT.name == 'detr-r50-unc.pth', 'Chọn đúng checkpoint R50 UNC từ TransVG.'
if not DETR_CHECKPOINT.is_file():
    if not AUTO_DOWNLOAD_DETR:
        raise FileNotFoundError(f'Gắn checkpoint TransVG và sửa DETR_CHECKPOINT: {DETR_CHECKPOINT}')
    import gdown
    DETR_CHECKPOINT.parent.mkdir(parents=True,exist_ok=True)
    partial = DETR_CHECKPOINT.with_suffix('.download')
    result = gdown.download(id=DETR_DOWNLOAD_ID,output=str(partial),quiet=False)
    if result is None or not partial.is_file():
        raise RuntimeError('Tải checkpoint thất bại. Tải từ link Drive trong hướng dẫn, chép vào pod và đổi DETR_CHECKPOINT.')
    os.replace(partial,DETR_CHECKPOINT)
detr_sha256 = file_sha256(DETR_CHECKPOINT)
assert cfg['detr_sha256'] == detr_sha256
if resume is not None:
    assert resume['config']['detr_sha256'] == detr_sha256, 'Checkpoint DETR khác lượt trước.'
print('DETR TransVG R50 UNC:',DETR_CHECKPOINT,'| SHA256:',detr_sha256)



In [ ]:
if resume is not None:
    saved_scales = resume['config'].get('train_resize_scales')
    if saved_scales is not None:
        assert tuple(saved_scales) == RESIZE_SCALES, 'Các mức resize đã đổi.'
    assert resume['config']['text_cache_spec'] == cache_spec

# Stem đến layer3 chỉ trích feature, không thuộc optimizer.
frozen_source = FrozenDETRFeatures(DETR_CHECKPOINT)
train_stem = nn.Sequential(*list(frozen_source.visual.backbone[0].body.children())[:-1])
del frozen_source
train_stem.requires_grad_(False).to(device).eval()


## 
Fusion/head cũ — thay input projection 2048→256 thành 256→256

In [ ]:
class PositionEncoding2D(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        assert d_model % 4 == 0
        self.d_model = d_model

    def forward(self, feature_map):
        _, _, H, W = feature_map.shape
        device = feature_map.device
        n = self.d_model // 4

        frequencies = 10000 ** (
            torch.arange(n, device=device, dtype=torch.float32) / n
        )

        y = torch.linspace(0, 2 * math.pi, H, device=device)
        x = torch.linspace(0, 2 * math.pi, W, device=device)

        y = y[:, None] / frequencies[None, :]
        x = x[:, None] / frequencies[None, :]

        pos_y = torch.cat([y.sin(), y.cos()], dim=-1)
        pos_x = torch.cat([x.sin(), x.cos()], dim=-1)

        pos_y = pos_y[:, None, :].expand(H, W, -1)
        pos_x = pos_x[None, :, :].expand(H, W, -1)

        position = torch.cat([pos_y, pos_x], dim=-1)

        return position.reshape(
            1, H * W, self.d_model
        ).to(dtype=feature_map.dtype)

class DETRCachedREC(nn.Module):
    def __init__(self, d_model=256):
        super().__init__()
        self.image_projection = nn.Conv2d(256, d_model, 1)
        self.image_position = PositionEncoding2D(d_model)
        self.text_projection = nn.Linear(768, d_model)
        self.modality_embedding = nn.Embedding(2, d_model)
        self.transformer = nn.Transformer(
            d_model=d_model, nhead=8, num_encoder_layers=6,
            num_decoder_layers=6, dim_feedforward=1024,
            dropout=0.1, activation='relu', batch_first=True,
        )
        self.object_query = nn.Embedding(1, d_model)
        self.box_head = nn.Sequential(
            nn.Linear(d_model,d_model), nn.ReLU(),
            nn.Linear(d_model,d_model), nn.ReLU(),
            nn.Linear(d_model,4), nn.Sigmoid(),
        )
    @torch.autocast(device_type='cuda', dtype=torch.float16)
    def forward(self, image_features, text_features, attention_mask):
        assert image_features.shape[1] == 256, 'Cần spatial memory DETR encoder.'
        x = image_features
        x = self.image_projection(x)
        image_tokens = x.flatten(2).transpose(1,2)
        image_tokens = image_tokens + self.image_position(x) + self.modality_embedding.weight[0]
        text_tokens = self.text_projection(text_features) + self.modality_embedding.weight[1]
        tokens = torch.cat([image_tokens, text_tokens], dim=1)
        image_padding = torch.zeros(image_features.shape[0], image_tokens.shape[1], dtype=torch.bool, device=image_features.device)
        padding = torch.cat([image_padding, ~attention_mask.bool()], dim=1)
        memory = self.transformer.encoder(tokens, src_key_padding_mask=padding)
        query = self.object_query.weight.unsqueeze(0).expand(image_features.shape[0],-1,-1)
        decoded = self.transformer.decoder(query, memory, memory_key_padding_mask=padding)
        return self.box_head(decoded[:,0,:])


class FineTuneREC(DETRCachedREC):
    def __init__(self):
        super().__init__()
        if warm is not None:
            self.load_state_dict(warm['model_state_dict'], strict=True)
        visual = FrozenDETRFeatures(DETR_CHECKPOINT).visual
        self.layer4 = visual.backbone[0].body.layer4.requires_grad_(True)
        self.pos = visual.backbone[1]
        self.skip_alpha = SKIP_ALPHA
        self.proj = visual.input_proj.requires_grad_(True)
        self.visual_encoder = visual.transformer.encoder.requires_grad_(True)
        text = AutoModel.from_pretrained(TEXT_MODEL, revision=resolved_revision, add_pooling_layer=False)
        self.text_tail = text.encoder.layer[-2:]

    @torch.autocast('cuda', dtype=torch.float16)
    def forward(self, image, text, attention_mask, return_features=False):
        x = self.proj(self.layer4(image))
        nested = nested_tensor_from_tensor_list(x)
        pos = self.pos(nested).to(x.dtype).flatten(2).permute(2,0,1)
        tokens = x.flatten(2).permute(2,0,1)
        memory = self.visual_encoder(tokens, src_key_padding_mask=nested.mask.flatten(1), pos=pos)
        x_post = memory.permute(1, 2, 0).reshape_as(x)
        image = x_post + self.skip_alpha * x if self.skip_alpha else x_post
        mask = (~attention_mask)[:,None,None,:].to(text.dtype) * torch.finfo(text.dtype).min
        for layer in self.text_tail:
            text = layer(text, attention_mask=mask)[0]
        boxes = super().forward(image, text, attention_mask)
        return (boxes, image, text) if return_features else boxes


base_model = FineTuneREC().to(device)
if resume is not None:
    base_model.load_state_dict(resume['model_state_dict'], strict=True)
del warm
model = base_model

if USE_CHANNELS_LAST:
    train_stem.to(memory_format=torch.channels_last)
    base_model.layer4.to(memory_format=torch.channels_last)


class RegionTextContrastive(nn.Module):
    def __init__(self):
        super().__init__()
        self.g = nn.Linear(256, HEAD_DIM)
        self.h = nn.Linear(768, HEAD_DIM)

    def forward(self, image_map, text, attention_mask, candidates):
        valid = [i for i, boxes in enumerate(candidates) if len(boxes) >= 2]
        if not valid:
            return image_map.new_zeros((), dtype=torch.float32), 0

        image_map = image_map[valid].float()
        height, width = image_map.shape[-2:]
        scale = image_map.new_tensor([width, height, width, height])
        boxes = [candidates[i].to(image_map.device) * scale for i in valid]
        regions = roi_align(image_map, boxes, output_size=1, sampling_ratio=2,
                            aligned=True).flatten(1)
        text = text[valid].float()
        mask = attention_mask[valid][..., None]
        query = (text * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)
        region_vec = F.normalize(self.g(regions), dim=-1)
        query_vec = F.normalize(self.h(query), dim=-1)

        # Mỗi tensor ứng viên đặt bbox đúng ở vị trí 0.
        groups = region_vec.split([len(b) for b in boxes])
        losses = [-(group @ q / TAU).log_softmax(dim=0)[0]
                  for group, q in zip(groups, query_vec)]
        return torch.stack(losses).mean(), len(valid)
contrastive_head = None
if CONTRASTIVE_ALPHA > 0:
    # Khởi tạo projection head không tiêu thụ RNG của pipeline train gốc.
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(SEED)
        contrastive_head = RegionTextContrastive().to(device)
    if resume is not None:
        contrastive_head.load_state_dict(resume['contrastive_state_dict'], strict=True)

In [ ]:
def load_image_pixels(sample):
    with Image.open(IMAGE_DIR / sample['file_name']) as image:
        assert image.size == (sample['width'], sample['height']), 'Metadata ảnh sai.'
        return image_transform(image.convert('RGB'))


def prepare_pixel_cache():
    unique_images = {
        row['image_id']: row for row in train_samples + val_samples + test_samples
    }
    image_ids = sorted(unique_images)
    shape = (len(image_ids), 3, *IMAGE_SIZE)
    spec = [annotation_sha, list(IMAGE_SIZE), str(IMAGE_DIR.resolve()),
            IMAGE_DATA_VERSION, Image.__version__, 'rgb_chw_pil_bilinear_v1']
    key = hashlib.sha256(json.dumps(spec).encode()).hexdigest()[:20]
    UINT8_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    path = UINT8_CACHE_ROOT / f'images_{key}.npy'
    if not path.is_file():
        required = math.prod(shape)
        if shutil.disk_usage(UINT8_CACHE_ROOT).free < required + 1024**3:
            raise RuntimeError(
                f'Cache ảnh cần {required / 1024**3:.2f} GiB và 1 GiB dự phòng. '
                'Đổi UINT8_CACHE_ROOT hoặc đặt USE_UINT8_IMAGE_CACHE=False.'
            )
        with TemporaryDirectory(dir=UINT8_CACHE_ROOT) as temporary:
            staging = Path(temporary) / path.name
            pixels = np.lib.format.open_memmap(
                staging, mode='w+', dtype=np.uint8, shape=shape,
            )
            for index, image_id in enumerate(tqdm(image_ids, desc='Cache pixel RGB uint8')):
                pixels[index] = load_image_pixels(unique_images[image_id]).numpy()
            pixels.flush()
            del pixels
            os.replace(staging, path)
    pixels = np.load(path, mmap_mode='r')
    if pixels.shape != shape or pixels.dtype != np.uint8:
        raise ValueError(f'Cache ảnh sai shape hoặc dtype: {path}')
    del pixels
    print(f'Cache pixel: {len(image_ids)} ảnh | {math.prod(shape) / 1024**3:.2f} GiB')
    return path, {image_id: index for index, image_id in enumerate(image_ids)}


pixel_cache_path, pixel_cache_index = None, {}
if USE_UINT8_IMAGE_CACHE:
    pixel_cache_path, pixel_cache_index = prepare_pixel_cache()



boxes_by_image = defaultdict(dict)
if CONTRASTIVE_ALPHA > 0:
    for row in train_samples:
        boxes_by_image[row['image_id']][tuple(row['bbox'])] = None


class RECDataset(Dataset):
    def __init__(self, rows, training=False):
        self.rows = rows
        self.training = training
        self.has_color = [bool(color_pattern.search(row['text'])) for row in rows]
        self.features = self.offsets = self.pixels = None

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        if self.features is None:
            self.features = np.load(CACHE_DIR / 'features.npy', mmap_mode='r')
            self.offsets = np.load(CACHE_DIR / 'offsets.npy', mmap_mode='r')
        s = self.rows[index]
        if pixel_cache_path is None:
            image = load_image_pixels(s)
        else:
            if self.pixels is None:
                self.pixels = np.load(pixel_cache_path, mmap_mode='r')
            image = torch.from_numpy(self.pixels[pixel_cache_index[s['image_id']]].copy())
        W, H = s['width'], s['height']
        x,y,w,h = s['bbox']
        box = torch.tensor([(x+w/2)/W, (y+h/2)/H, w/W, h/H], dtype=torch.float32)
        idx = text_to_index[s['text']]
        text = torch.from_numpy(self.features[self.offsets[idx]:self.offsets[idx+1]].copy())

        candidates = torch.empty(0, 4, dtype=torch.float32)
        if self.training and s['image_id'] in boxes_by_image:
            image_boxes = boxes_by_image[s['image_id']]
            if len(image_boxes) > 1:
                positive = tuple(s['bbox'])
                negatives = [b for b in image_boxes if b != positive]
                xywh = torch.tensor([positive, *negatives], dtype=torch.float32)
                candidates = (box_convert(xywh, 'xywh', 'xyxy') /
                              xywh.new_tensor([W, H, W, H])).clamp(0, 1)
        return image, text, box, self.has_color[index], candidates

def collate(batch):
    images, texts, boxes, has_color, candidates = zip(*batch)
    lengths = torch.tensor([len(t) for t in texts])
    text = nn.utils.rnn.pad_sequence(texts, batch_first=True)
    mask = torch.arange(text.shape[1])[None,:] < lengths[:,None]
    return dict(images=torch.stack(images), text_features=text,
                attention_mask=mask, boxes=torch.stack(boxes),
                has_color=torch.tensor(has_color, dtype=torch.bool), candidates=candidates)

train_generator = torch.Generator().manual_seed(SEED)
def worker_init(worker_id):
    seed = torch.initial_seed() % (2**32)
    random.seed(seed)
    np.random.seed(seed)

def make_loader(rows, training, keep_workers=True):
    options = dict(batch_size=BATCH_SIZE, shuffle=training,
                   num_workers=NUM_WORKERS, pin_memory=True,
                   collate_fn=collate, worker_init_fn=worker_init,
                   persistent_workers=keep_workers and NUM_WORKERS > 0)
    if NUM_WORKERS > 0:
        options['prefetch_factor'] = PREFETCH_FACTOR
    if training:
        options['generator'] = train_generator
    return DataLoader(RECDataset(rows, training), **options)

train_loader = make_loader(train_samples, True)
val_loader = make_loader(val_samples, False)
test_loader = make_loader(test_samples, False, keep_workers=False)
print('Train batches:', len(train_loader), '| Val batches:', len(val_loader))

In [ ]:
def grounding_loss(pred, target):
    pred, target = pred.float(), target.float()
    l1 = F.l1_loss(pred, target, reduction='none').sum(-1).mean()
    p = box_convert(pred, in_fmt='cxcywh', out_fmt='xyxy')
    t = box_convert(target, in_fmt='cxcywh', out_fmt='xyxy')
    return L1_WEIGHT*l1 + GIOU_WEIGHT*generalized_box_iou_loss(p,t,reduction='mean')

@torch.no_grad()
def matched_iou(pred, target):
    # IoU từng cặp, không tạo ma trận B×B rồi lấy đường chéo.
    p = box_convert(pred.float(), in_fmt='cxcywh', out_fmt='xyxy')
    t = box_convert(target.float(), in_fmt='cxcywh', out_fmt='xyxy')
    wh = (torch.minimum(p[:,2:],t[:,2:])-torch.maximum(p[:,:2],t[:,:2])).clamp(min=0)
    intersection = wh.prod(-1)
    union = (p[:,2:]-p[:,:2]).clamp(min=0).prod(-1) + (t[:,2:]-t[:,:2]).clamp(min=0).prod(-1) - intersection
    return intersection / union.clamp(min=1e-7)

trainable_params = [p for p in base_model.parameters() if p.requires_grad]
tail_names = ('layer4.', 'proj.', 'visual_encoder.', 'text_tail.')
param_groups = [
    dict(params=[p for n,p in base_model.named_parameters() if n.startswith(tail_names)], lr=1e-5, name='pretrained'),
    dict(params=[p for n,p in base_model.named_parameters() if not n.startswith(tail_names)], lr=OTHER_LR, name='fusion_head')
]
if contrastive_head is not None:
    head_params = list(contrastive_head.parameters())
    trainable_params.extend(head_params)
    param_groups.append(dict(params=head_params, lr=OTHER_LR, name='contrastive_head'))
optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
scaler = torch.amp.GradScaler('cuda')

if resume is not None:
    optimizer.load_state_dict(resume['optimizer_state_dict'])
    scheduler.load_state_dict(resume['scheduler_state_dict'])
    scaler.load_state_dict(resume['scaler_state_dict'])
    train_generator.set_state(resume['train_generator_state'])
    random.setstate(resume['python_rng_state'])
    ns = resume['numpy_rng_state']
    np.random.set_state((ns[0], np.array(ns[1], dtype=np.uint32), ns[2], ns[3], ns[4]))
    torch.set_rng_state(resume['torch_rng_state'])
    torch.cuda.set_rng_state(resume['cuda_rng_states'][0], device=device)

image_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
image_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def run_epoch(loader, training, description):
    model.train(training)
    if contrastive_head is not None:
        contrastive_head.train(training)
    loss_parts = torch.zeros(3, dtype=torch.float64, device=device)
    # Cộng dồn trên GPU; chỉ lấy số liệu và kiểm tra loss khi log/cuối epoch.
    totals = torch.zeros(4, dtype=torch.float64, device=device)
    color_totals = torch.zeros(3, dtype=torch.float64, device=device)
    seen = 0
    checked_grad = False
    torch.cuda.synchronize()
    started = time.perf_counter()
    progress = tqdm(loader, desc=description, mininterval=2.0)
    scales = []
    if training:
        while len(scales) < len(loader):
            group = list(RESIZE_SCALES)
            random.shuffle(group)
            scales.extend(group)

    for step, batch in enumerate(progress, 1):
        candidates = batch.pop('candidates')
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        images = batch['images'].float().div_(255.0)
        images = images.sub_(image_mean).div_(image_std).half()
        with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
            scale = scales[step - 1] if training else 1.0
            size = tuple(round(dim * scale) for dim in IMAGE_SIZE)
            if size != IMAGE_SIZE:
                images = F.interpolate(images.float(), size=size, mode='bilinear',
                                       align_corners=False, antialias=True).half()
            if USE_CHANNELS_LAST:
                images = images.contiguous(memory_format=torch.channels_last)
            image_features = train_stem(images).half()
            if USE_CHANNELS_LAST:
                image_features = image_features.contiguous(memory_format=torch.channels_last)

        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            text, mask = batch['text_features'], batch['attention_mask']
            if training and contrastive_head is not None and any(len(b) > 1 for b in candidates):
                pred, image_map, text_tail = model(image_features, text, mask, return_features=True)
                contrastive, valid = contrastive_head(image_map, text_tail, mask, candidates)
                del image_map, text_tail
            else:
                pred = model(image_features, text, mask)
                contrastive, valid = pred.new_zeros((), dtype=torch.float32), 0
            bbox_loss = grounding_loss(pred, batch['boxes'])
            loss = bbox_loss + CONTRASTIVE_ALPHA * contrastive
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                if not checked_grad:
                    assert base_model.image_projection.weight.grad is not None, 'Projection thiếu gradient!'
                    assert base_model.box_head[0].weight.grad is not None, 'Head không có gradient!'
                    checked_grad = True
                nn.utils.clip_grad_norm_(trainable_params, 1.0)
                scaler.step(optimizer)
                scaler.update()
        iou = matched_iou(pred.detach(), batch['boxes'])
        n = len(iou)
        totals += torch.stack([loss.detach().double()*n, iou.double().sum(), (iou>=0.5).double().sum(), (~torch.isfinite(loss.detach())).double()])
        if not training:
            has_color = batch['has_color']
            color_totals += torch.stack([
                has_color.sum(),
                ((iou >= 0.5) & has_color).sum(),
                (iou.double() * has_color).sum(),
            ])
        loss_parts[0] += bbox_loss.detach().double() * n
        loss_parts[1] += contrastive.detach().double() * valid
        loss_parts[2] += valid
        seen += n
        if step % LOG_EVERY == 0 or step == len(loader):
            values = (totals/seen).tolist()
            if values[3] > 0 or not all(math.isfinite(v) for v in values[:3]):
                raise RuntimeError('Có loss/metric NaN hoặc Inf từ lần log trước. Dừng, không ghi đè checkpoint tốt. Nạp lại last/best trước khi thử tiếp.')
            elapsed = time.perf_counter()-started
            progress.set_postfix(loss=f'{values[0]:.4f}', mIoU=f'{values[1]:.4f}', acc50=f'{values[2]:.4f}', samples_s=f'{seen/elapsed:.1f}')
    assert seen > 0
    torch.cuda.synchronize()
    elapsed = time.perf_counter()-started
    values = (totals/seen).tolist()
    if values[3] > 0 or not all(math.isfinite(v) for v in values[:3]):
        raise RuntimeError('Loss/metric NaN hoặc Inf ở cuối epoch; không lưu checkpoint.')
    metrics = dict(loss=values[0], miou=values[1], acc50=values[2],
                   minutes=elapsed / 60, samples_s=seen / elapsed)
    bbox_total, contrastive_total, valid_count = loss_parts.tolist()
    metrics.update(bbox_loss=bbox_total / seen,
                   contrastive_loss=contrastive_total / valid_count if valid_count else None,
                   contrastive_n=int(valid_count))
    if not training:
        color_n, color_correct, color_iou = color_totals.tolist()
        metrics.update(
            color_n=int(color_n),
            color_acc50=color_correct / color_n if color_n else None,
            color_miou=color_iou / color_n if color_n else None,
        )
    return metrics

## Train, validation và lưu checkpoint theo epoch

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BEST_PATH = OUTPUT_DIR / 'best_checkpoint.pth'
LAST_PATH = OUTPUT_DIR / 'last_checkpoint.pth'
CONFIG = dict(
    stage='rec_finetune_layer4_text2_skip_contrastive', annotation_sha256=annotation_sha,
    contrastive_alpha=CONTRASTIVE_ALPHA, contrastive_tau=TAU, contrastive_head_dim=HEAD_DIM,
    skip_alpha=SKIP_ALPHA, color_words=list(COLOR_WORDS),
    train_resize_scales=list(RESIZE_SCALES),
    text_model=TEXT_MODEL, text_revision=resolved_revision, text_cache_spec=cache_spec,
    detr_model=DETR_MODEL, detr_checkpoint=str(DETR_CHECKPOINT), detr_sha256=detr_sha256,
    transvg_visual_commit=TRANSVG_VISUAL_COMMIT,
    image_size_hw=list(IMAGE_SIZE), batch_size=BATCH_SIZE,
    image_pipeline='online_frozen_stem',
    uint8_image_cache=USE_UINT8_IMAGE_CACHE, channels_last=USE_CHANNELS_LAST,
    loss_weights=[L1_WEIGHT,GIOU_WEIGHT], initial_other_lr=OTHER_LR,
    trainable_image_layers=['layer4','input_proj','encoder'], trainable_text_layers=[10,11],
    d_model=256, encoder_layers=6, decoder_layers=6, nhead=8,
    dim_feedforward=1024, dropout=0.1, activation='relu', object_queries=1,
    num_workers=NUM_WORKERS, prefetch_factor=PREFETCH_FACTOR,
    seed=SEED, cudnn_benchmark=True, torch_version=str(torch.__version__),
)

def save_checkpoint(path, epoch):
    ns = np.random.get_state()
    payload = dict(
        epoch=epoch, model_state_dict=base_model.state_dict(),
        contrastive_state_dict=contrastive_head.state_dict() if contrastive_head is not None else None,
        optimizer_state_dict=optimizer.state_dict(), scheduler_state_dict=scheduler.state_dict(),
        scaler_state_dict=scaler.state_dict(), baseline_metrics=baseline_metrics,
        best_acc50=best_acc50, best_epoch=best_epoch, history=history, config=CONFIG,
        train_generator_state=train_generator.get_state(),
        python_rng_state=random.getstate(),
        numpy_rng_state=[ns[0],ns[1].tolist(),ns[2],ns[3],ns[4]],
        torch_rng_state=torch.get_rng_state(), cuda_rng_states=torch.cuda.get_rng_state_all(),
    )
    temp = path.with_suffix('.tmp')
    torch.save(payload, temp)
    os.replace(temp, path)

session_started = time.perf_counter()
if resume is None:
    if BEST_PATH.exists() or LAST_PATH.exists():
        raise FileExistsError('Output đã có checkpoint. Đổi OUTPUT_DIR hoặc đặt RESUME_PATH đúng; không ghi đè lượt cũ.')
    baseline_metrics = run_epoch(val_loader, False, 'Validation skip tại mốc khởi tạo epoch 10')
    print('Validation skip tại mốc khởi tạo epoch 10:', baseline_metrics)
    best_acc50, best_epoch = baseline_metrics['acc50'], 10
    history, start_epoch = [], 11
    save_checkpoint(BEST_PATH, 10)
    save_checkpoint(LAST_PATH, 10)
else:
    baseline_metrics = resume['baseline_metrics']
    history = resume['history']
    best_acc50, best_epoch = resume['best_acc50'], resume['best_epoch']
    start_epoch = resume['epoch'] + 1
    source_best = RESUME_PATH.parent / 'best_checkpoint.pth'
    assert source_best.is_file(), 'Resume cần best_checkpoint.pth bên cạnh last để giữ best toàn bộ quá trình.'
    best_check = torch.load(source_best,map_location='cpu',weights_only=True)
    assert best_check['config'] == resume['config'], 'Best và last khác cấu hình.'
    assert best_check['epoch'] == best_epoch and best_check['best_acc50'] == best_acc50
    if best_check['config'] != CONFIG:
        # Chuyển metadata best cũ sang cấu hình resume mới, giữ nguyên weights.
        best_check['config'] = CONFIG
        temporary_best = BEST_PATH.with_suffix('.tmp')
        torch.save(best_check, temporary_best)
        os.replace(temporary_best, BEST_PATH)
    elif source_best.resolve() != BEST_PATH.resolve():
        shutil.copy2(source_best, BEST_PATH)
    del best_check
    del resume
    gc.collect()
    print('Tiếp tục từ epoch', start_epoch, '| Best epoch:', best_epoch)

for epoch in range(start_epoch, EPOCHS+1):
    epoch_started = time.perf_counter()
    lrs = {g['name']:g['lr'] for g in optimizer.param_groups}
    print(f'\nEpoch {epoch}/{EPOCHS}, LR={lrs}')
    train_metrics = run_epoch(train_loader, True, f'Epoch {epoch} | Train REC')
    val_metrics = run_epoch(val_loader, False, f'Epoch {epoch} | Validation')
    scheduler.step()
    history.append(dict(epoch=epoch, train=train_metrics, val=val_metrics, learning_rates_used=lrs))
    improved = val_metrics['acc50'] > best_acc50
    if improved:
        best_acc50, best_epoch = val_metrics['acc50'], epoch
        save_checkpoint(BEST_PATH, epoch)
    save_checkpoint(LAST_PATH, epoch)
    (OUTPUT_DIR/'history.json').write_text(json.dumps(dict(
        baseline=baseline_metrics, best_epoch=best_epoch, best_acc50=best_acc50,
        config=CONFIG, history=history), ensure_ascii=False, indent=2))
    epoch_seconds = time.perf_counter()-epoch_started
    print('Train:', train_metrics, '\nVal:', val_metrics)
    print(f'Best Acc@0.5={best_acc50:.4f} ở epoch {best_epoch}; cả epoch + lưu: {epoch_seconds/60:.1f} phút')
    if MAX_SESSION_HOURS is not None:
        remaining = MAX_SESSION_HOURS*3600 - (time.perf_counter()-session_started)
        if remaining < epoch_seconds + NEXT_EPOCH_RESERVE_MIN*60:
            print('Kết thúc lượt này để giữ output. Tiếp tục từ last_checkpoint.pth ở phiên sau.')
            break

print('Best:', best_acc50, '| Epoch:', best_epoch)
print('Tăng so epoch 10:', round((best_acc50-baseline_metrics['acc50'])*100,3), 'điểm phần trăm')
print('Output:', OUTPUT_DIR)

In [ ]:
if RUN_TEST:
    selected = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
    base_model.load_state_dict(selected['model_state_dict'], strict=True)
    del selected
    evaluations = {'skip': run_epoch(test_loader, False, 'TestA | skip')}
    if BASELINE_BEST_PATH is not None:
        baseline = torch.load(Path(BASELINE_BEST_PATH), map_location='cpu', weights_only=True)
        baseline_config = baseline['config']
        assert baseline_config['stage'] == 'rec_finetune_layer4_text2', 'Chọn checkpoint model gốc.'
        for key in ('annotation_sha256', 'detr_sha256', 'text_cache_spec', 'image_size_hw'):
            assert baseline_config[key] == CONFIG[key], f'Baseline khác cấu hình ở {key}.'
        try:
            base_model.load_state_dict(baseline['model_state_dict'], strict=True)
            base_model.skip_alpha = 0.0
            evaluations['baseline'] = run_epoch(test_loader, False, 'TestA | baseline')
        finally:
            del baseline
            selected = torch.load(BEST_PATH, map_location='cpu', weights_only=True)
            base_model.load_state_dict(selected['model_state_dict'], strict=True)
            base_model.skip_alpha = SKIP_ALPHA
            del selected

    (OUTPUT_DIR / 'test_metrics.json').write_text(
        json.dumps(evaluations, indent=2), encoding='utf-8'
    )
    for name, metrics in evaluations.items():
        print(name, metrics)
else:
    print('RUN_TEST=False. Bật sau khi chốt thí nghiệm.')
